In [6]:
import torch
import torch.nn as nn
from transformers import AutoModel
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
import torch
import torch.nn as nn
from transformers.modeling_outputs import CausalLMOutputWithPast
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from icecream import ic
from transformers import get_cosine_schedule_with_warmup
from tqdm.autonotebook import tqdm
import warnings

In [7]:
MODEL_NAME = "google/paligemma2-3b-mix-224"
paligemma = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    use_fact = True
    
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [3]:
for name, param in paligemma.named_parameters():
    # if 'vision' not in name and 'language' not in name:
    print(name)

model.vision_tower.vision_model.embeddings.patch_embedding.weight
model.vision_tower.vision_model.embeddings.patch_embedding.bias
model.vision_tower.vision_model.embeddings.position_embedding.weight
model.vision_tower.vision_model.encoder.layers.0.layer_norm1.weight
model.vision_tower.vision_model.encoder.layers.0.layer_norm1.bias
model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj.weight
model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj.bias
model.vision_tower.vision_model.encoder.layers.0.self_attn.v_proj.weight
model.vision_tower.vision_model.encoder.layers.0.self_attn.v_proj.bias
model.vision_tower.vision_model.encoder.layers.0.self_attn.q_proj.weight
model.vision_tower.vision_model.encoder.layers.0.self_attn.q_proj.bias
model.vision_tower.vision_model.encoder.layers.0.self_attn.out_proj.weight
model.vision_tower.vision_model.encoder.layers.0.self_attn.out_proj.bias
model.vision_tower.vision_model.encoder.layers.0.layer_norm2.weight
model.vision_tower.

In [33]:
def collate_fn(examples):
    texts  = []
    images = []

    for ex in examples:
        image = ex["images"][0]
        if image.mode != "RGB":
            image = image.convert("RGB")
        image = image.resize((384, 384))

        question = ex["question"].strip()
        options  = ex["answers"]                         # e.g. ['3', '0', '6', '4', '5']

        # correct_answer is the raw value → find its index → convert to letter
        answer_idx = options.index(ex["correct_answer"]) # e.g. '3' is at index 0 → 'A'
        answer     = chr(65 + answer_idx)                # 'A', 'B', 'C', ...

        options_text = "\n".join([f"{chr(65+i)}. {opt}" for i, opt in enumerate(options)])

        prompt = (
            f"<image> Question: {question}\n"
            f"Options:\n{options_text}\n"
            f"Answer:"
        )
        texts.append((prompt, answer))
        images.append(image)

    eos        = processor.tokenizer.eos_token
    full_texts = [f"{q}\n{a}{eos}" for q, a in texts]   # FIX 2: append EOS so model learns to stop

    # FIX: only one processor call needed now
    batch_input = processor(
        text=full_texts, images=images, return_tensors="pt", padding=True,
    )

    labels = batch_input["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == 257152] = -100  # image token id

    for i in range(len(texts)):
        full_len   = (batch_input["attention_mask"][i] == 1).sum().item()
        
        # Detect if processor added a trailing \n after EOS
        newline_id  = processor.tokenizer.encode("\n", add_special_tokens=False)[-1]  # 108
        last_tok    = batch_input["input_ids"][i][full_len - 1].item()
        has_trailing = (last_tok == newline_id)

        a_ids       = processor.tokenizer(texts[i][1], add_special_tokens=False)["input_ids"]
        answer_len  = len(a_ids) + 1 + (1 if has_trailing else 0)  # A + EOS + maybe \n
        mask_until  = full_len - answer_len

        if mask_until <= 0:
            labels[i, :] = -100
            continue

        labels[i, :mask_until] = -100

    batch_input["labels"] = labels
    return batch_input

In [34]:
dataset = load_dataset("array/SAT-v2", split="train")
dataset = dataset.train_test_split(0.2)
train_dataset, test_dataset = dataset['train'], dataset['test']

train_loader = DataLoader(train_dataset, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, collate_fn=collate_fn)


In [36]:
batch = next(iter(train_loader))

In [38]:
ids    = batch["input_ids"][0]
labels = batch["labels"][0]

# Print only the unmasked (answer) portion
unmasked = [(i, ids[i].item(), labels[i].item()) 
            for i in range(len(ids)) if labels[i].item() != -100]
print("idx | token_id | decoded")
for idx, tok, lbl in unmasked:
    print(f"{idx:4d} | {tok:8d} | {repr(processor.tokenizer.decode([tok]))}")


idx | token_id | decoded
 309 |   235280 | 'A'
 310 |        1 | '<eos>'
 311 |      108 | '\n'


In [22]:
processor.tokenizer.decode([      2,  10825, 235292,  30919,   1089,
            576,    573,   9113,    575,    573,   5528,   5265,    674,    692,
            798,   2076,   1443,    575,    573,   2257,   5265,   8509,    774,
           1024,   3464,  12568, 235336,    108,   6928, 235292,    108, 235280,
         235265,    793,   9113,   8509,    108, 235305, 235265,  89685,    729,
           8509,   2731,    578,   3024,    774,    573,   7909,    575,    573,
           1370,   5265,    108,   1261, 235292,    108, 235280,      1,    108])

'<bos> Question: Were any of the objects in the initial frame that you can still see in the second frame moved from their original positions?\nOptions:\nA. no objects moved\nB. Statue was moved left and away from the camera in the first frame\nAnswer:\nA<eos>\n'

In [29]:
processor.tokenizer.decode([108])

'\n'

In [39]:
batch

{'input_ids': tensor([[257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152,
         257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152,
         257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152,
         257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152,
         257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152,
         257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152,
         257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152,
         257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152,
         257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152,
         257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152,
         257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152,
         257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152, 257152,
         25715

In [3]:
dataset

Dataset({
    features: ['images', 'question', 'answers', 'correct_answer', 'question_type'],
    num_rows: 172384
})

In [ ]:
for name, _ in paligemma.language_model.named_modules():
    if "q_proj" in name:
        print(name)
        break

layers.0.self_attn.q_proj


In [6]:
for name, _ in paligemma.named_parameters():
    if 'language' in name:
        print(name)

model.language_model.embed_tokens.weight
model.language_model.layers.0.self_attn.q_proj.weight
model.language_model.layers.0.self_attn.k_proj.weight
model.language_model.layers.0.self_attn.v_proj.weight
model.language_model.layers.0.self_attn.o_proj.weight
model.language_model.layers.0.mlp.gate_proj.weight
model.language_model.layers.0.mlp.up_proj.weight
model.language_model.layers.0.mlp.down_proj.weight
model.language_model.layers.0.input_layernorm.weight
model.language_model.layers.0.post_attention_layernorm.weight
model.language_model.layers.0.pre_feedforward_layernorm.weight
model.language_model.layers.0.post_feedforward_layernorm.weight
model.language_model.layers.1.self_attn.q_proj.weight
model.language_model.layers.1.self_attn.k_proj.weight
model.language_model.layers.1.self_attn.v_proj.weight
model.language_model.layers.1.self_attn.o_proj.weight
model.language_model.layers.1.mlp.gate_proj.weight
model.language_model.layers.1.mlp.up_proj.weight
model.language_model.layers.1.mlp.

In [8]:
target_modules = []

for name, module in paligemma.named_modules():
    if (
        "language_model" in name
        and any(x in name for x in ["q_proj", "k_proj", "v_proj", "o_proj"])
    ):
        target_modules.append(name)

target_modules

['model.language_model.layers.0.self_attn.q_proj',
 'model.language_model.layers.0.self_attn.k_proj',
 'model.language_model.layers.0.self_attn.v_proj',
 'model.language_model.layers.0.self_attn.o_proj',
 'model.language_model.layers.1.self_attn.q_proj',
 'model.language_model.layers.1.self_attn.k_proj',
 'model.language_model.layers.1.self_attn.v_proj',
 'model.language_model.layers.1.self_attn.o_proj',
 'model.language_model.layers.2.self_attn.q_proj',
 'model.language_model.layers.2.self_attn.k_proj',
 'model.language_model.layers.2.self_attn.v_proj',
 'model.language_model.layers.2.self_attn.o_proj',
 'model.language_model.layers.3.self_attn.q_proj',
 'model.language_model.layers.3.self_attn.k_proj',
 'model.language_model.layers.3.self_attn.v_proj',
 'model.language_model.layers.3.self_attn.o_proj',
 'model.language_model.layers.4.self_attn.q_proj',
 'model.language_model.layers.4.self_attn.k_proj',
 'model.language_model.layers.4.self_attn.v_proj',
 'model.language_model.layers.4

In [11]:
for name, p in paligemma.named_parameters():   # [FIX] named_parameters(), not parameters()
    if 'multi_modal_projector' in name:              #       p is a tensor, not a string
        p.requires_grad = True
        print("nn")

nn
nn


In [9]:
for p in paligemma.parameters():
    p.requires_grad = False

# target_modules=r"language_model.*\.(q_proj|k_proj|v_proj|o_proj)$"
lora_config = LoraConfig(
        r=32,
        lora_alpha=128,
        target_modules=target_modules, # this will do all the models
        lora_dropout=0.05,
        bias="none",
    )

paligemma = get_peft_model(paligemma, lora_config)
paligemma.print_trainable_parameters()

trainable params: 12,779,520 || all params: 3,045,021,936 || trainable%: 0.4197


In [7]:
type(paligemma.language_model)

transformers.models.gemma2.modeling_gemma2.Gemma2Model

In [6]:
lora_config = LoraConfig(
            r=32,
            lora_alpha=128,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "multi_modal_projector.linear"], # this will do all the models
            lora_dropout=0.05,
            bias="none",
        )
# self.model_base = get_peft_model(model_base, lora_config)
# self.model_base.print_trainable_parameters()

paligemma = get_peft_model(paligemma, lora_config)
paligemma.print_trainable_parameters()

trainable params: 18,862,080 || all params: 3,051,104,496 || trainable%: 0.6182


In [7]:
lr = 1e-4
weight_decay = 0.01
epochs = 3
warum_ratio = 0.05
device = "cuda" if torch.cuda.is_available() else "cpu"

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, paligemma.parameters()),
    lr=lr,
    weight_decay=weight_decay,
)

num_training_steps = epochs * len(train_loader)
num_warmup_steps = int(warum_ratio * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

# criterion = nn.CrossEntropyLoss()
paligemma = paligemma.to(device)

In [8]:
total_params = sum(
    p.numel() for p in paligemma.parameters() if p.requires_grad
)
print(f"Total trainable params: {total_params}")
warnings.filterwarnings("ignore", message="You are passing both `text` and `images` to `PaliGemmaProcessor`.*")

for epoch in range(epochs):

    paligemma.train()

    total_loss = 0.0

    for batch in tqdm(train_loader):

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        batch["pixel_values"] = batch["pixel_values"].to(torch.bfloat16)

        optimizer.zero_grad()
        warnings.filterwarnings("ignore", message="You are passing both `text` and `images` to `PaliGemmaProcessor`.*")
        outputs = paligemma(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    paligemma.eval()

    val_loss = 0.0

    with torch.no_grad():

        for batch in test_loader:

            batch = {
                k: v.to(device)
                for k, v in batch.items()
            }

            batch["pixel_values"] = batch["pixel_values"].to(torch.bfloat16)
            warnings.filterwarnings("ignore", message="You are passing both `text` and `images` to `PaliGemmaProcessor`.*")

            outputs = paligemma(**batch)

            val_loss += outputs.loss.item()

    avg_val_loss = val_loss / len(test_loader)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {avg_train_loss:.4f} "
        f"Val Loss: {avg_val_loss:.4f}"
    )

Total trainable params: 18862080


  0%|          | 0/137907 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# for name, param in paligemma.named_parameters():
#     if param.requires_grad:
#         print(name)

In [ ]:
# batch = next(iter(train_loader))
# batch.keys()

In [ ]:
# ls = [ 2,   9413, 235292,  65288,    573,
#            8761,  12568, 235269,    603,   8426,  37559,  15949,    887,  17279,
#            3037,    577,    573,   2731,    689,   1833,    576,   2656,   5354,
#            9205,    675,   1536, 231220, 235336,    108,   6928, 235292,    108,
#          235280, 235265,   2731,    108, 235305, 235265,   1833,    108,   1261,
#          235292,    108,   1672,    108]
# processor.tokenizer.decode([1672,
#           108])